This notebook is to test accesing the database that was created when indexing the data, similar to how the process would be within the RAG pipeline. We would index and access the data independently.

In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors.db'
)

In [4]:
query = "Can I still join the course?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

## Implementing rag with sqlitesearch vector search

All we need to update to do this is the index that the assistant uses.

In [5]:
import os
from dotenv import dotenv_values

from rag_helper import VectorRAGBase
from ingest import load_faq_data, build_index

from openai import OpenAI

In [6]:
secrets_dir = os.path.expanduser("~/Documents/.secrets/llm-zoomcamp/")

config = {
    **dotenv_values(secrets_dir + "/.env.openai"),
}

api_key = config.get("OPENAI_API_KEY")
openai_client = OpenAI(api_key=api_key)

In [7]:
vector_assistant = VectorRAGBase(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [8]:
vector_assistant.rag('How does the course work?')

'The course is run with a “live” cohort, not self-paced, if you want a certificate. You follow the course during the live run, submit your project while submissions are open, and then peer-review 3 capstone projects after submitting your own.'

In [9]:
vs_index.close()